# MSPR - Préparation et nettoyage des datasets

Ce notebook regroupe les traitements réalisés sur **4 jeux de données** du projet.  
L'idée est de présenter les traitements **dataset par dataset**.

Datasets traités :
1. INSEE - recensement / emploi
2. Élections
3. RNA - associations
4. Délinquance Rennes Métropole


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

PROJECT_DIR = Path(".")
DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_CLEAN = PROJECT_DIR / "data" / "clean"

DATA_CLEAN.mkdir(parents=True, exist_ok=True)

print("Dossier courant :", PROJECT_DIR.resolve())
print("Dossier raw :", DATA_RAW.resolve())
print("Dossier clean :", DATA_CLEAN.resolve())

Dossier courant : C:\Users\oizel\Documents\MSPR Data
Dossier raw : C:\Users\oizel\Documents\MSPR Data\data\raw
Dossier clean : C:\Users\oizel\Documents\MSPR Data\data\clean


## 1. Dataset INSEE - recensement / emploi

Dans cette partie, on traite deux besoins :
- agréger la population estimée par **année** et **département de résidence**
- préparer un fichier communal plus léger sur l'**emploi / population active**


In [ ]:
# Fichier recensement
path_ficdep = DATA_RAW / "Ficdep22.csv"
path_ficdep

WindowsPath('data/raw/Ficdep22.csv')

In [ ]:
# Lecture par morceaux pour éviter de charger tout le fichier d'un coup
acc = {}

for chunk in pd.read_csv(
    path_ficdep,
    sep=";",
    encoding="latin1",
    engine="python",
    usecols=["AN_RECENS", "DEP_RES_24", "POND"],
    chunksize=300_000,
    low_memory=True,
    on_bad_lines="skip"
):
    chunk = chunk.copy()
    chunk["DEP_RES_24"] = chunk["DEP_RES_24"].astype(str).str.zfill(2)
    chunk = chunk[chunk["DEP_RES_24"].str.fullmatch(r"\d{2}", na=False)]
    chunk["POND"] = pd.to_numeric(chunk["POND"], errors="coerce")
    chunk = chunk.dropna(subset=["AN_RECENS", "DEP_RES_24", "POND"])

    grouped = chunk.groupby(["AN_RECENS", "DEP_RES_24"], as_index=False)["POND"].sum()

    for _, row in grouped.iterrows():
        key = (int(row["AN_RECENS"]), row["DEP_RES_24"])
        acc[key] = acc.get(key, 0) + float(row["POND"])

df_dep = pd.DataFrame(
    [{"annee": annee, "dep": dep, "population_estimee": valeur} for (annee, dep), valeur in acc.items()]
).sort_values(["annee", "dep"]).reset_index(drop=True)

df_dep.head()

In [ ]:
# Export du fichier agrégé
out_pop = DATA_CLEAN / "pop_dep_res_annee.csv"
df_dep.to_csv(out_pop, index=False, encoding="utf-8")
out_pop

WindowsPath('data/clean/pop_dep_res_annee.csv')

In [ ]:
# Fichier emploi / population active 2022

from pathlib import Path

# 👉 Mets ton vrai chemin ici
path_emploi = Path("C:/Users/oizel/Documents/MSPR Data/base-cc-emploi-pop-active-2022.csv")

cols_utiles = [
    "CODGEO",
    "P22_POP1564",
    "P22_ACT1564",
    "P22_ACTOCC1564",
    "P22_CHOM1564",
]

# Vérification simple (évite plantage brutal)
if not path_emploi.exists():
    raise FileNotFoundError(f"Fichier introuvable : {path_emploi}")

df_emploi = pd.read_csv(
    path_emploi,
    sep=";",
    usecols=cols_utiles,
    dtype={"CODGEO": str},
    encoding="latin1"
)

# Renommage des colonnes
df_emploi = df_emploi.rename(columns={
    "CODGEO": "code_insee",
    "P22_POP1564": "pop_1564",
    "P22_ACT1564": "actifs_1564",
    "P22_ACTOCC1564": "actifs_occupes_1564",
    "P22_CHOM1564": "chomeurs_1564",
})

# Formatage du code INSEE
df_emploi["code_insee"] = df_emploi["code_insee"].str.zfill(5)

df_emploi.head()

,code_insee,pop_1564,actifs_1564,actifs_occupes_1564,chomeurs_1564
0,01001,525.000000,431.000000,403.000000,28.000000
1,01002,166.000000,133.000000,131.000000,2.000000
2,01004,9752.373817,7479.600207,6473.855566,1005.744641
3,01005,1228.753771,999.457257,936.133860,63.323397
4,01006,72.000000,51.000000,46.000000,5.000000


In [ ]:
out_emploi = DATA_CLEAN / "emploi_pop_active_commune_2022_clean.csv"
df_emploi.to_csv(out_emploi, index=False, encoding="utf-8")
out_emploi

WindowsPath('data/clean/emploi_pop_active_commune_2022_clean.csv')

## 2. Dataset élections

Ici, l'objectif est surtout de :
- charger le fichier des résultats électoraux
- vérifier sa structure
- repérer les colonnes utiles pour la suite du projet


In [ ]:
path_elections = PROJECT_DIR / "Données" / "resultat_election_presidentiel_complet_bloc.csv"

df_elections = pd.read_csv(path_elections, sep=";")
df_elections.head()

,CODE_ELECTION,DATE_ELECTION,NUMERO_TOUR,NUMERO_BV,NB_INSCRITS,NB_EXPRIMES,NOM_CANDIDAT,NB_VOIX,BLOC_POLITIQUE
0,P02,2002,1,11.0,1200,795,MEGRET,5.0,AUTRE
1,P02,2002,1,11.0,1200,795,LEPAGE,26.0,CTR
2,P02,2002,1,11.0,1200,795,GLUCKSTEIN,8.0,EXG
3,P02,2002,1,11.0,1200,795,BAYROU,75.0,CTR
4,P02,2002,1,11.0,1200,795,CHIRAC,201.0,DRT


In [ ]:
# Contrôle rapide de la structure
df_elections.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11926 entries, 0 to 11925
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CODE_ELECTION   11926 non-null  object 
 1   DATE_ELECTION   11926 non-null  object 
 2   NUMERO_TOUR     11926 non-null  int64  
 3   NUMERO_BV       9162 non-null   float64
 4   NB_INSCRITS     11926 non-null  int64  
 5   NB_EXPRIMES     11926 non-null  int64  
 6   NOM_CANDIDAT    11926 non-null  object 
 7   NB_VOIX         11926 non-null  float64
 8   BLOC_POLITIQUE  11926 non-null  object 
dtypes: float64(2), int64(3), object(4)
memory usage: 838.7+ KB


In [ ]:
# Colonnes disponibles
print(df_elections.columns.tolist())

['CODE_ELECTION', 'DATE_ELECTION', 'NUMERO_TOUR', 'NUMERO_BV', 'NB_INSCRITS', 'NB_EXPRIMES', 'NOM_CANDIDAT', 'NB_VOIX', 'BLOC_POLITIQUE']


In [ ]:
# Aperçu ciblé des colonnes utiles pour notre projet
colonnes_utiles_elections = [
    "CODE_ELECTION",
    "DATE_ELECTION",
    "NUMERO_TOUR",
    "NUMERO_BV",
    "NB_INSCRITS",
    "NB_EXPRIMES",
    "NOM_CANDIDAT",
    "NB_VOIX",
    "BLOC_POLITIQUE",
]

df_elections[colonnes_utiles_elections].head(10)

,CODE_ELECTION,DATE_ELECTION,NUMERO_TOUR,NUMERO_BV,NB_INSCRITS,NB_EXPRIMES,NOM_CANDIDAT,NB_VOIX,BLOC_POLITIQUE
0,P02,2002,1,11.0,1200,795,MEGRET,5.0,AUTRE
1,P02,2002,1,11.0,1200,795,LEPAGE,26.0,CTR
2,P02,2002,1,11.0,1200,795,GLUCKSTEIN,8.0,EXG
3,P02,2002,1,11.0,1200,795,BAYROU,75.0,CTR
4,P02,2002,1,11.0,1200,795,CHIRAC,201.0,DRT
5,P02,2002,1,11.0,1200,795,LE PEN,60.0,EXD
6,P02,2002,1,11.0,1200,795,TAUBIRA,31.0,GCH
7,P02,2002,1,11.0,1200,795,SAINT-JOSSE,5.0,DIV
8,P02,2002,1,11.0,1200,795,MAMERE,89.0,AUTRE
9,P02,2002,1,11.0,1200,795,JOSPIN,136.0,GCH


## 3. Dataset RNA - associations

Dans cette partie, on filtre le fichier RNA du département 35 pour ne conserver
que les communes de Rennes Métropole, puis on prépare une sortie agrégée par commune
et par grande catégorie.


In [ ]:
path_rna = DATA_RAW / "rna_import_20250901_dpt_35.csv"

df_rna = pd.read_csv(path_rna, sep=";", dtype=str, low_memory=False)
print(df_rna.shape)
df_rna.head(3)

(13021, 23)


,id,id_ex,siret,gestion,date_creat,date_publi,nature,groupement,titre,objet,objet_social1,objet_social2,adr1,adr2,adr3,adrs_codepostal,libcom,dir_civilite,siteweb,observation,position,rup_mi,maj_time
0,351S0351000089,0351000089,NaN,351S,0001-01-01,1914-05-17,D,S,AMICALE SAINTE JEANNE D'ARC - SAINT JOSEPH.,Etablir entre tous les anciens eleves des rela...,015070,000000,"8, rue jeanne d'arc",NaN,NaN,35133,FOUGERES,PM,NaN,Reprise auto => date publication création au ...,A,NaN,2005-12-06 17:02:56
1,351S0351000090,0351000090,NaN,351S,0001-01-01,0001-01-01,D,S,LA JEANNE D'ARC,Favoriser l' Etude de la musique,006030,000000,12 rue de bonabry,NaN,NaN,35133,FOUGERES,PM,NaN,NaN,A,NaN,2005-12-06 17:03:00
2,351S0351000092,0351000092,NaN,351S,0001-01-01,0001-01-01,D,S,SOCIETE DES AMIS DES ECOLES PUBLIQUES D4ANTRAI...,Developper et encourager les oeuvres scolaires...,015005,000000,Cercle antrainais,NaN,NaN,35560,ANTRAIN,PM,NaN,NaN,A,NaN,2005-12-06 17:02:56


In [ ]:
# On garde d'abord les lignes du département 35
df_rna_35 = df_rna[df_rna["adrs_codepostal"].str.startswith("35", na=False)].copy()

print("Avant filtrage :", df_rna.shape)
print("Après filtrage département 35 :", df_rna_35.shape)

Avant filtrage : (13021, 23)
Après filtrage département 35 : (10522, 23)


In [ ]:
# Liste des communes de Rennes Métropole présentes dans notre traitement
communes_rm = [
    "ACIGNE", "BECHEREL", "BETTON", "BRUZ", "CESSON-SEVIGNE", "CHANTEPIE",
    "CHARTRES-DE-BRETAGNE", "CHASNE-SUR-ILLET", "LA CHAPELLE-DES-FOUGERETZ",
    "LA CHAPELLE-THOUARAULT", "LE RHEU", "LE VERGER", "L'HERMITAGE", "MONTGERMONT",
    "NOYAL-CHATILLON-SUR-SEICHE", "ORGERES", "PACÉ", "PARTHENAY-DE-BRETAGNE",
    "PONT-PEAN", "RENNES", "ROMILLE", "SAINT-ARMEL", "SAINT-GILLES",
    "SAINT-GREGOIRE", "SAINT-JACQUES-DE-LA-LANDE", "THORIGNE-FOUILLARD", "VEZIN-LE-COQUET"
]

df_rna_rm = df_rna_35[df_rna_35["libcom"].isin(communes_rm)].copy()
print("Après filtrage Rennes Métropole :", df_rna_rm.shape)

Après filtrage Rennes Métropole : (3903, 23)


In [ ]:
# Colonnes souhaitées pour le projet
colonnes_rna = [
    "id", "titre", "objet", "date_creation", "libcom", "adrs_codepostal",
    "code_objet_1", "code_objet_2"
]

# On garde seulement les colonnes réellement présentes
colonnes_disponibles = [col for col in colonnes_rna if col in df_rna_rm.columns]
colonnes_manquantes = [col for col in colonnes_rna if col not in df_rna_rm.columns]

print("Colonnes disponibles :", colonnes_disponibles)
print("Colonnes manquantes :", colonnes_manquantes)

df_rna_clean = df_rna_rm[colonnes_disponibles].copy()
df_rna_clean.head()

Colonnes disponibles : ['id', 'titre', 'objet', 'libcom', 'adrs_codepostal']
Colonnes manquantes : ['date_creation', 'code_objet_1', 'code_objet_2']


,id,titre,objet,libcom,adrs_codepostal
1458,352S0352001787,ETONNANTS VOYAGEURS,"La promotion, l'organisation, la creation et l...",RENNES,35000
2004,352S0352002642,ELECTRO - PHONE,Soutenir et faire valoir les pratiques artisti...,RENNES,35000
2152,353P0353000061,ASSOCIATION SPORTIVE DE L'ECOLE NORMALE MIXTE ...,ORGANISER ET CONTROLER LA PRATIQUE DES SPORTS ...,RENNES,35000
2155,353P0353000198,FOOTBALL CLUB RENNAIS,O,RENNES,35000
2157,353P0353000211,UNION SPORTIVE SAINT MARTIN,PROMOUVOIR ET ORGANISER EN PROLONGEMENT DE L'E...,RENNES,35000


In [ ]:
# Regroupement simple des objets associatifs
# On ne crée la catégorie que si la colonne code_objet_1 existe

if "code_objet_1" in df_rna_clean.columns:
    df_rna_clean["famille_code"] = df_rna_clean["code_objet_1"].astype(str).str[:3]

    mapping = {
        "001": "Culture",
        "002": "Sport",
        "003": "Loisirs",
        "004": "Education",
        "005": "Solidarite",
        "006": "Sante",
        "007": "Environnement",
    }

    df_rna_clean["categorie_principale"] = (
        df_rna_clean["famille_code"].map(mapping).fillna("Autres")
    )

else:
    df_rna_clean["famille_code"] = None
    df_rna_clean["categorie_principale"] = "Non renseigne"

df_rna_clean[["libcom", "categorie_principale"]].head()

,libcom,categorie_principale
1458,RENNES,Non renseigne
2004,RENNES,Non renseigne
2152,RENNES,Non renseigne
2155,RENNES,Non renseigne
2157,RENNES,Non renseigne


In [ ]:
# Agrégation finale par commune et catégorie
df_commune_categorie = (
    df_rna_clean
    .groupby(["libcom", "categorie_principale"], as_index=False)
    .size()
    .rename(columns={"size": "nb_associations"})
)

df_commune_categorie.head()

,libcom,categorie_principale,nb_associations
0,ACIGNE,Non renseigne,51
1,BECHEREL,Non renseigne,22
2,BETTON,Non renseigne,88
3,BRUZ,Non renseigne,159
4,CESSON-SEVIGNE,Non renseigne,199


In [ ]:
out_rna = DATA_CLEAN / "rna_rm_commune_categorie.csv"
df_commune_categorie.to_csv(out_rna, index=False, encoding="utf-8")
out_rna

WindowsPath('data/clean/rna_rm_commune_categorie.csv')

## 4. Dataset délinquance - Rennes Métropole

Dans cette partie, on filtre le fichier sur les communes de Rennes Métropole,
on corrige quelques problèmes d'encodage, puis on prépare une versio


In [ ]:
SRC_CSV = PROJECT_DIR / "donnee-data.gouv-2024-geographie2025-produit-le2025-06-04.csv"

RENMet_INSEE = [
    "35001", "35002", "35004", "35005", "35006", "35007", "35008", "35009",
    "35010", "35011", "35013", "35014", "35015", "35016", "35018", "35019",
    "35020", "35021", "35022", "35023", "35024", "35025", "35026", "35027",
    "35028", "35029", "35030", "35031", "35032", "35033", "35034", "35035",
    "35036", "35037", "35038", "35039", "35040", "35041", "35042", "35238"
]

header = pd.read_csv(SRC_CSV, sep=";", nrows=0, encoding="utf-8")
print(header.columns.tolist())

['CODGEO_2025', 'annee', 'indicateur', 'unite_de_compte', 'nombre', 'taux_pour_mille', 'est_diffuse', 'insee_pop', 'insee_pop_millesime', 'insee_log', 'insee_log_millesime', 'complement_info_nombre', 'complement_info_taux']


In [ ]:
# Petite fonction pour corriger certains caractères mal encodés
def fix_mojibake(s):
    if not isinstance(s, str):
        return s
    if "Ã" in s or "Â" in s:
        try:
            return s.encode("latin1").decode("utf-8")
        except Exception:
            return s
    return s

def fix_dataframe_text(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].map(fix_mojibake)
    return df

In [ ]:
# Lecture par morceaux puis filtrage sur Rennes Métropole
chunks = []

for chunk in pd.read_csv(
    SRC_CSV,
    sep=";",
    chunksize=200_000,
    encoding="utf-8"
):
    chunk = chunk[chunk["CODGEO_2025"].astype(str).isin(RENMet_INSEE)]
    if not chunk.empty:
        chunks.append(chunk)

df_rm = pd.concat(chunks, ignore_index=True)
df_rm.shape

C:\Users\oizel\AppData\Local\Temp\ipykernel_2308\764501460.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\oizel\AppData\Local\Temp\ipykernel_2308\764501460.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\oizel\AppData\Local\Temp\ipykernel_2308\764501460.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\oizel\AppData\Local\Temp\ipykernel_2308\764501460.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\oizel\AppData\Local\Temp\ipykernel_2308\764501460.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\oizel\AppData\Local\Temp\i

(4995, 13)

In [ ]:
# Correction de texte et contrôle rapide
df_rm = fix_dataframe_text(df_rm)

for col in df_rm.select_dtypes(include="object").columns:
    if df_rm[col].astype(str).str.contains("Ã|Â", regex=True).any():
        print("Colonne encore suspecte :", col)

In [ ]:
# Nettoyage de la colonne nombre
df_rm["nombre"] = pd.to_numeric(df_rm["nombre"], errors="coerce")

moyenne_par_indicateur = (
    df_rm.groupby("indicateur", dropna=False)["nombre"]
    .mean()
    .reset_index()
    .rename(columns={"nombre": "moyenne_nombre_indicateur"})
)

df_rm = df_rm.merge(moyenne_par_indicateur, on="indicateur", how="left")

mask_nombre_vide = df_rm["nombre"].isna()
df_rm.loc[mask_nombre_vide, "nombre"] = df_rm.loc[mask_nombre_vide, "moyenne_nombre_indicateur"]
df_rm["nombre_impute"] = False
df_rm.loc[mask_nombre_vide, "nombre_impute"] = True

print("Valeurs encore vides :", df_rm["nombre"].isna().sum())

Valeurs encore vides : 0


In [ ]:
# Conversion de quelques types utiles pour la suite
colonnes_decimal = ["taux_pour_mille", "complement_info_nombre", "complement_info_taux"]

for col in colonnes_decimal:
    if col in df_rm.columns:
        df_rm[col] = (
            df_rm[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .replace("nan", np.nan)
            .astype(float)
        )

if "annee" in df_rm.columns:
    df_rm["annee"] = pd.to_numeric(df_rm["annee"], errors="coerce")

df_rm.head()

,CODGEO_2025,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux,moyenne_nombre_indicateur,nombre_impute
0,35001,2016,Violences physiques intrafamiliales,Victime,5.000000,0.752445,diff,6645,2016,2859,2016,NaN,NaN,64.977273,False
1,35001,2016,Violences physiques hors cadre familial,Victime,9.000000,1.354402,diff,6645,2016,2859,2016,NaN,NaN,88.530000,False
2,35001,2016,Violences sexuelles,Victime,53.681159,NaN,ndiff,6645,2016,2859,2016,1.431034,0.576946,53.681159,True
3,35001,2016,Vols avec armes,Infraction,0.000000,0.000000,diff,6645,2016,2859,2016,NaN,NaN,1.516340,False
4,35001,2016,Vols violents sans arme,Infraction,18.227092,NaN,ndiff,6645,2016,2859,2016,0.837209,0.179680,18.227092,True


In [ ]:
out_delinquance = DATA_CLEAN / "delinquance_35.csv"
df_rm.to_csv(out_delinquance, sep=";", index=False, encoding="utf-8")
out_delinquance

WindowsPath('data/clean/delinquance_35.csv')

## 5. Conclusion

Chaque étape est décomposé en 3 sous-étapes: :
- un dataset
- un traitement principal
- un export propre dans `data/clean`


